In [6]:
import cv2
import mediapipe as mp
import urllib.request
import os

# 1. Download the hand model automatically


model_url = "https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task"
model_path = "hand_landmarker.task"

if not os.path.exists(model_path):
    print("Downloading hand model...")
    urllib.request.urlretrieve(model_url, model_path)
    print("Model downloaded successfully!")
else:
    print("Hand model already exists!")



# 2. Create MediaPipe Hand Detector


BaseOptions = mp.tasks.BaseOptions
HandLandmarker = mp.tasks.vision.HandLandmarker
HandLandmarkerOptions = mp.tasks.vision.HandLandmarkerOptions
VisionRunningMode = mp.tasks.vision.RunningMode

options = HandLandmarkerOptions(
    base_options=BaseOptions(model_asset_path=model_path),
    running_mode=VisionRunningMode.VIDEO,
    num_hands=2,
    min_hand_detection_confidence=0.5,
    min_hand_presence_confidence=0.5,
    min_tracking_confidence=0.5
)

detector = HandLandmarker.create_from_options(options)

print("Hand detector created successfully!")


# ==========================================
# 3. Open Laptop Camera
# ==========================================

cap = cv2.VideoCapture(0)

if not cap.isOpened():
    print("Camera could not be opened.")
else:
    print("Camera opened successfully!")
    print("Press Q to quit.")


# ==========================================
# 4. Detect Hands
# ==========================================

frame_timestamp = 0

while cap.isOpened():

    success, frame = cap.read()

    if not success:
        print("Could not read camera frame.")
        break

    # Flip the image like a mirror
    frame = cv2.flip(frame, 1)

    # Convert BGR to RGB
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    # Convert to MediaPipe image
    mp_image = mp.Image(
        image_format=mp.ImageFormat.SRGB,
        data=rgb_frame
    )

    # Increase timestamp
    frame_timestamp += 1

    # Detect hands
    result = detector.detect_for_video(
        mp_image,
        frame_timestamp
    )


    # ==========================================
    # 5. Draw Hand Landmarks
    # ==========================================

    if result.hand_landmarks:

        for hand_landmarks in result.hand_landmarks:

            # Draw 21 points
            for landmark in hand_landmarks:

                x = int(landmark.x * frame.shape[1])
                y = int(landmark.y * frame.shape[0])

                cv2.circle(
                    frame,
                    (x, y),
                    6,
                    (0, 255, 0),
                    -1
                )

            # Draw connections between points
            connections = [
                (0,1), (1,2), (2,3), (3,4),
                (0,5), (5,6), (6,7), (7,8),
                (5,9), (9,10), (10,11), (11,12),
                (9,13), (13,14), (14,15), (15,16),
                (13,17), (17,18), (18,19), (19,20),
                (0,17)
            ]

            for start, end in connections:

                x1 = int(hand_landmarks[start].x * frame.shape[1])
                y1 = int(hand_landmarks[start].y * frame.shape[0])

                x2 = int(hand_landmarks[end].x * frame.shape[1])
                y2 = int(hand_landmarks[end].y * frame.shape[0])

                cv2.line(
                    frame,
                    (x1, y1),
                    (x2, y2),
                    (0, 255, 0),
                    2
                )


    # ==========================================
    # 6. Show Camera
    # ==========================================

    cv2.imshow("Hand Detector", frame)


    # Press Q to quit
    if cv2.waitKey(1) & 0xFF == ord("q"):
        break


# ==========================================
# 7. Close Everything
# ==========================================

cap.release()
cv2.destroyAllWindows()
detector.close()

print("Camera closed.")

Hand model already exists!
Hand detector created successfully!
Camera opened successfully!
Press Q to quit.
Camera closed.


In [3]:
import mediapipe as mp

print(mp.__version__)
print(mp.__file__)

1.0.1
C:\Users\DELL\anaconda3\Lib\site-packages\mediapipe\__init__.py
